In [3]:
from google.colab import files
from pathlib import Path
import zipfile
import pandas as pd

uploaded = files.upload()

zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_names:
    raise ValueError("Please upload All_GTFS_CSV_Files.zip.")

PROJECT = Path("/content/RouteRanker")
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
REPORTS = PROJECT / "reports" / "module2"

for folder in [RAW, PROCESSED, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(RAW)

matches = list(RAW.rglob("Vapi_GTFS/routes.csv"))

if not matches:
    raise FileNotFoundError(
        "Vapi_GTFS/routes.csv was not found. "
        "Check that you selected the correct ZIP."
    )

DATA = matches[0].parent
print("Dataset found:", DATA)

Saving All_GTFS_CSV_Files-1.zip to All_GTFS_CSV_Files-1.zip
Dataset found: /content/RouteRanker/data/raw/Vapi_GTFS


In [4]:
import pandas as pd
from IPython.display import display

tables = {}

for name in ["routes", "trips", "stop_times", "stops", "calendar"]:
    tables[name] = pd.read_csv(
        DATA / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    print(f"\n{name}.csv: {len(tables[name]):,} records")
    display(tables[name].head(3))

routes = tables["routes"]
trips = tables["trips"]
stop_times = tables["stop_times"]
stops = tables["stops"]
calendar = tables["calendar"]

print("\nLoading completed.")


routes.csv: 22 records


,route_id,agency_id,route_short_name,route_long_name,route_type
0,1868,VCBS,11,"Vapi Town Post office - Meril, Welspun Company...",3
1,1867,VCBS,10,"Vapi Town Post office - Dabhel Check Post, Kun...",3
2,1829,VCBS,1,Vapi Railway Station (East) - G.I.D.C. Third P...,3



trips.csv: 461 records


,route_id,service_id,trip_id,trip_headsign,shape_id
0,1868,daily,trip_1868_0,Bhagyodaya Society,shape_1868
1,1868,daily,trip_1868_1,Bhagyodaya Society,shape_1868
2,1868,daily,trip_1868_2,Bhagyodaya Society,shape_1868



stop_times.csv: 2,144 records


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,shape_dist_traveled
0,trip_1868_0,12:27:00,12:27:00,4803,1,0.0
1,trip_1868_0,12:29:04,12:29:04,4804,2,767.5010089761968
2,trip_1868_0,12:31:48,12:31:48,4805,3,1915.9958495084684



stops.csv: 91 records


,stop_id,stop_name,stop_lat,stop_lon
0,4803,Alok Industries,20.416004,72.908704
1,4758,Amba Mata Temple,20.381,72.9292
2,4748,Anavil Hall,20.370258,72.898163



calendar.csv: 1 records


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,daily,1,1,1,1,1,1,1,20260101,20271231



Loading completed.


In [5]:
# Use a fixed date so the results can be reproduced.
SERVICE_DATE = pd.Timestamp("2026-10-31")
weekday = SERVICE_DATE.day_name().lower()

start_dates = pd.to_datetime(
    calendar["start_date"],
    format="%Y%m%d",
    errors="coerce"
)

end_dates = pd.to_datetime(
    calendar["end_date"],
    format="%Y%m%d",
    errors="coerce"
)

if start_dates.isna().any() or end_dates.isna().any():
    raise ValueError("Invalid calendar dates need investigation.")

# Find services operating on the selected date.
operating = (
    start_dates.le(SERVICE_DATE)
    & end_dates.ge(SERVICE_DATE)
    & calendar[weekday].eq("1")
)

active_services = set(
    calendar.loc[operating, "service_id"]
)

# Apply service additions and cancellations, if available.
exceptions_path = DATA / "calendar_dates.csv"

if exceptions_path.exists():
    exceptions = pd.read_csv(
        exceptions_path,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    selected = exceptions[
        exceptions["date"].eq(
            SERVICE_DATE.strftime("%Y%m%d")
        )
    ]

    active_services.update(
        selected.loc[
            selected["exception_type"].eq("1"),
            "service_id"
        ]
    )

    active_services.difference_update(
        selected.loc[
            selected["exception_type"].eq("2"),
            "service_id"
        ]
    )

active_trips = trips[
    trips["service_id"].isin(active_services)
].copy()

if active_trips.empty:
    raise ValueError("No scheduled trips found for this date.")

print("Service date:", SERVICE_DATE.date())
print("Day:", SERVICE_DATE.day_name())
print(
    "Day type:",
    "Weekend" if SERVICE_DATE.dayofweek >= 5 else "Weekday"
)
print("Active services:", sorted(active_services))
print("Distinct scheduled trips:", active_trips["trip_id"].nunique())

display(active_trips.head(3))

Service date: 2026-10-31
Day: Saturday
Day type: Weekend
Active services: ['daily']
Distinct scheduled trips: 461


,route_id,service_id,trip_id,trip_headsign,shape_id
0,1868,daily,trip_1868_0,Bhagyodaya Society,shape_1868
1,1868,daily,trip_1868_1,Bhagyodaya Society,shape_1868
2,1868,daily,trip_1868_2,Bhagyodaya Society,shape_1868


In [6]:
# Select stop visits belonging to active trips.
active_stop_times = stop_times[
    stop_times["trip_id"].isin(active_trips["trip_id"])
].copy()

# Convert stop sequence to numbers for correct ordering.
active_stop_times["sequence_number"] = pd.to_numeric(
    active_stop_times["stop_sequence"],
    errors="coerce"
)

sequence = active_stop_times["sequence_number"]

if (
    sequence.isna()
    | sequence.lt(0)
    | sequence.mod(1).ne(0)
).any():
    raise ValueError("Invalid stop sequences need investigation.")

if active_stop_times.duplicated(
    ["trip_id", "sequence_number"]
).any():
    raise ValueError("Repeated stop sequences need investigation.")

# Find the first stop of each trip.
first_stops = (
    active_stop_times
    .sort_values(["trip_id", "sequence_number"])
    .drop_duplicates("trip_id")
    [["trip_id", "departure_time"]]
    .copy()
)

# Accept GTFS hours above 24, such as 25:30:00.
valid_time = first_stops["departure_time"].str.fullmatch(
    r"\d{2,}:[0-5]\d:[0-5]\d"
)

parts = (
    first_stops["departure_time"]
    .where(valid_time)
    .str.extract(r"^(\d+):(\d{2}):(\d{2})$")
    .apply(pd.to_numeric, errors="coerce")
)

first_stops["departure_seconds"] = (
    parts[0] * 3600 + parts[1] * 60 + parts[2]
)

# Connect departure features to trips.
trip_features = active_trips.merge(
    first_stops,
    on="trip_id",
    how="left",
    validate="one_to_one"
)

trip_features["service_hour"] = (
    trip_features["departure_seconds"] // 3600
).astype("Int64")

trip_features["clock_hour"] = trip_features["service_hour"] % 24

trip_features["minute"] = (
    (trip_features["departure_seconds"] % 3600) // 60
).astype("Int64")

def classify_period(hour):
    if pd.isna(hour):
        return pd.NA
    if 5 <= hour < 12:
        return "Morning"
    if 12 <= hour < 17:
        return "Afternoon"
    if 17 <= hour < 21:
        return "Evening"
    return "Night"

trip_features["time_period"] = (
    trip_features["clock_hour"].apply(classify_period)
)

trip_features["service_date"] = SERVICE_DATE.strftime("%Y-%m-%d")
trip_features["day_of_week"] = SERVICE_DATE.day_name()
trip_features["day_type"] = (
    "Weekend" if SERVICE_DATE.dayofweek >= 5 else "Weekday"
)

print("Trip records:", len(trip_features))
print(
    "Missing or invalid first departure times:",
    trip_features["departure_seconds"].isna().sum()
)

display(trip_features[[
    "route_id",
    "trip_id",
    "departure_time",
    "service_hour",
    "clock_hour",
    "minute",
    "time_period",
    "day_type"
]].head(10))

print("Time features created.")

Trip records: 461
Missing or invalid first departure times: 0


,route_id,trip_id,departure_time,service_hour,clock_hour,minute,time_period,day_type
0,1868,trip_1868_0,12:27:00,12,12,27,Afternoon,Weekend
1,1868,trip_1868_1,13:02:00,13,13,2,Afternoon,Weekend
2,1868,trip_1868_2,13:25:00,13,13,25,Afternoon,Weekend
3,1868,trip_1868_3,13:35:00,13,13,35,Afternoon,Weekend
4,1868,trip_1868_4,13:58:00,13,13,58,Afternoon,Weekend
5,1868,trip_1868_5,14:08:00,14,14,8,Afternoon,Weekend
6,1868,trip_1868_6,14:31:00,14,14,31,Afternoon,Weekend
7,1868,trip_1868_7,14:41:00,14,14,41,Afternoon,Weekend
8,1868,trip_1868_8,15:04:00,15,15,4,Afternoon,Weekend
9,1868,trip_1868_9,15:14:00,15,15,14,Afternoon,Weekend


Time features created.


In [7]:
# Check identifiers before joining tables.
if routes["route_id"].duplicated().any():
    raise ValueError("Duplicate route IDs need investigation.")

if active_trips["trip_id"].duplicated().any():
    raise ValueError("Duplicate trip IDs need investigation.")

if not active_trips["route_id"].isin(routes["route_id"]).all():
    raise ValueError("Some trips reference unknown routes.")

if not active_stop_times["stop_id"].isin(stops["stop_id"]).all():
    raise ValueError("Some stop visits reference unknown stops.")

# 1. Count distinct active trips per route.
trip_counts = (
    active_trips.groupby("route_id")["trip_id"]
    .nunique()
    .reset_index(name="scheduled_trip_count")
)

# 2. Connect stop visits to their routes.
route_stop_visits = active_stop_times.merge(
    active_trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left",
    validate="many_to_one"
)

# 3. Count distinct stops per route.
stop_counts = (
    route_stop_visits.groupby("route_id")["stop_id"]
    .nunique()
    .reset_index(name="distinct_stop_count")
)

# 4. Combine with route information.
route_profiles = (
    routes
    .merge(trip_counts, on="route_id", how="left")
    .merge(stop_counts, on="route_id", how="left")
)

# No active trips on the selected date means a verified zero.
route_profiles["scheduled_trip_count"] = (
    route_profiles["scheduled_trip_count"]
    .fillna(0)
    .astype(int)
)

route_profiles["distinct_stop_count"] = (
    route_profiles["distinct_stop_count"].astype("Int64")
)

route_profiles["service_date"] = SERVICE_DATE.strftime("%Y-%m-%d")

# 5. Verify totals.
assert route_profiles["scheduled_trip_count"].sum() == (
    active_trips["trip_id"].nunique()
)

print("Routes:", len(route_profiles))
print(
    "Total scheduled trips:",
    route_profiles["scheduled_trip_count"].sum()
)

display(route_profiles[[
    "route_id",
    "route_short_name",
    "scheduled_trip_count",
    "distinct_stop_count",
    "service_date"
]])

print("Route counts verified.")

Routes: 22
Total scheduled trips: 461


,route_id,route_short_name,scheduled_trip_count,distinct_stop_count,service_date
0,1868,11,31,13,2026-10-31
1,1867,10,16,4,2026-10-31
2,1829,1,19,5,2026-10-31
3,1890,6,9,7,2026-10-31
4,1833,3,21,5,2026-10-31
5,1830,1,51,2,2026-10-31
6,1881,5,9,5,2026-10-31
7,1882,5,9,2,2026-10-31
8,1889,6,9,7,2026-10-31
9,1831,2,24,8,2026-10-31


Route counts verified.


In [8]:
# ROUTERANKER — MODULE 2: FEATURE ENGINEERING
# Run this entire program in ONE Google Colab cell.

from google.colab import files
from pathlib import Path
from IPython.display import display
import pandas as pd
import numpy as np
import zipfile
import shutil
import json

# --------------------------------------------------
# 1. SETTINGS AND DATA UPLOAD
# --------------------------------------------------

SERVICE_DATE = pd.Timestamp("2026-10-31")

PROJECT = Path("/content/RouteRanker_Module2")
RAW = PROJECT / "data/raw"
OUTPUT = PROJECT / "deliverables"

RAW.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Upload All_GTFS_CSV_Files.zip")
uploaded = files.upload()

archives = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if len(archives) != 1:
    raise ValueError("Please upload exactly one GTFS ZIP archive.")

with zipfile.ZipFile(archives[0]) as archive:
    # Reject paths that would escape the extraction folder.
    root = RAW.resolve()
    for member in archive.infolist():
        target = (RAW / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError("Unsafe path found in ZIP.")
    archive.extractall(RAW)

matches = list(RAW.rglob("Vapi_GTFS/routes.csv"))

if not matches:
    raise FileNotFoundError("Vapi_GTFS/routes.csv was not found.")

DATA = matches[0].parent

def load(name, required=True):
    for extension in [".csv", ".txt"]:
        path = DATA / f"{name}{extension}"
        if path.exists():
            return pd.read_csv(
                path,
                dtype="string",
                keep_default_na=False,
                encoding="utf-8-sig"
            )
    if required:
        raise FileNotFoundError(f"Missing file: {name}")
    return None

tables = {
    name: load(name)
    for name in ["routes", "trips", "stop_times", "stops"]
}

for name in [
    "calendar", "calendar_dates", "shapes",
    "frequencies", "feed_info"
]:
    table = load(name, required=False)
    if table is not None:
        tables[name] = table

routes = tables["routes"]
trips = tables["trips"]
stop_times = tables["stop_times"]
stops = tables["stops"]

quality = []

def note(check, count, action):
    quality.append({
        "check": check,
        "affected_records": int(count),
        "action_or_interpretation": action
    })

# --------------------------------------------------
# 2. INITIAL QUALITY CHECKS
# --------------------------------------------------

inventory = []

for name, df in tables.items():
    empty = df.apply(lambda col: col.str.strip().eq("")).sum()

    inventory.append({
        "file": name,
        "records": len(df),
        "columns": ", ".join(df.columns),
        "empty_cells": int(empty.sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

    note(
        f"{name}: exact duplicate rows",
        df.duplicated().sum(),
        "Recorded; raw files preserved."
    )

for name, key in [
    ("routes", "route_id"),
    ("trips", "trip_id"),
    ("stops", "stop_id")
]:
    df = tables[name]

    if df[key].str.strip().eq("").any():
        raise ValueError(f"{name}: blank {key}; investigate first.")

    if df[key].duplicated().any():
        raise ValueError(f"{name}: duplicate {key}; investigate first.")

for child, key, parent in [
    ("trips", "route_id", "routes"),
    ("stop_times", "trip_id", "trips"),
    ("stop_times", "stop_id", "stops")
]:
    bad = ~tables[child][key].isin(tables[parent][key])

    note(
        f"{child}.{key}: unmatched references",
        bad.sum(),
        "Must be resolved before feature calculation."
    )

    if bad.any():
        raise ValueError(f"Broken relationship: {child}.{key}")

for column, lower, upper in [
    ("stop_lat", -90, 90),
    ("stop_lon", -180, 180)
]:
    values = pd.to_numeric(stops[column], errors="coerce")
    bad = values.isna() | ~values.between(lower, upper)
    note(
        f"stops: invalid {column}",
        bad.sum(),
        "Reported; stop coordinates are not used for distance estimates."
    )

# Frequency feeds need separate expansion logic.
frequencies = tables.get("frequencies")
if frequencies is not None and not frequencies.empty:
    raise ValueError(
        "This feed contains frequency-based service. "
        "This program intentionally stops rather than counting "
        "frequency templates as individual departures."
    )

# --------------------------------------------------
# 3. SELECT ACTIVE SERVICES
# --------------------------------------------------

active_services = set()
calendar = tables.get("calendar")

if calendar is not None:
    if calendar["service_id"].duplicated().any():
        raise ValueError("Duplicate calendar service IDs.")

    date_values = {}

    for column in ["start_date", "end_date"]:
        valid_format = calendar[column].str.fullmatch(r"\d{8}")
        parsed = pd.to_datetime(
            calendar[column].where(valid_format),
            format="%Y%m%d",
            errors="coerce"
        )

        if parsed.isna().any():
            raise ValueError(f"Invalid calendar {column}.")

        date_values[column] = parsed

    if (
        date_values["start_date"] > date_values["end_date"]
    ).any():
        raise ValueError("Calendar start date occurs after end date.")

    weekdays = [
        "monday", "tuesday", "wednesday", "thursday",
        "friday", "saturday", "sunday"
    ]

    for day in weekdays:
        if not calendar[day].isin(["0", "1"]).all():
            raise ValueError(f"Invalid calendar flag: {day}")

    day = SERVICE_DATE.day_name().lower()
    active = (
        date_values["start_date"].le(SERVICE_DATE)
        & date_values["end_date"].ge(SERVICE_DATE)
        & calendar[day].eq("1")
    )
    active_services.update(calendar.loc[active, "service_id"])

exceptions = tables.get("calendar_dates")

if exceptions is not None:
    parsed = pd.to_datetime(
        exceptions["date"].where(
            exceptions["date"].str.fullmatch(r"\d{8}")
        ),
        format="%Y%m%d",
        errors="coerce"
    )

    if parsed.isna().any():
        raise ValueError("Invalid calendar exception date.")

    if not exceptions["exception_type"].isin(["1", "2"]).all():
        raise ValueError("Invalid calendar exception type.")

    if exceptions.duplicated(["service_id", "date"]).any():
        raise ValueError("Duplicate calendar exception keys.")

    today = exceptions[parsed.eq(SERVICE_DATE)]

    active_services.update(
        today.loc[today["exception_type"].eq("1"), "service_id"]
    )
    active_services.difference_update(
        today.loc[today["exception_type"].eq("2"), "service_id"]
    )

if calendar is None and exceptions is None:
    raise ValueError("No calendar information available.")

known_services = set()
for name in ["calendar", "calendar_dates"]:
    if name in tables:
        known_services.update(tables[name]["service_id"])

if not trips["service_id"].isin(known_services).all():
    raise ValueError("Trips reference unknown service IDs.")

active_trips = trips[
    trips["service_id"].isin(active_services)
].copy()

if active_trips.empty:
    raise ValueError("No active trips for the selected service date.")

# --------------------------------------------------
# 4. TIME FEATURES AND STOP PATTERNS
# --------------------------------------------------

st = stop_times[
    stop_times["trip_id"].isin(active_trips["trip_id"])
].copy()

st["sequence"] = pd.to_numeric(
    st["stop_sequence"], errors="coerce"
)

invalid_sequence = (
    st["sequence"].isna()
    | st["sequence"].lt(0)
    | st["sequence"].mod(1).ne(0)
)

if invalid_sequence.any():
    raise ValueError("Invalid stop sequences.")

if st.duplicated(["trip_id", "sequence"]).any():
    raise ValueError("Repeated trip/stop-sequence keys.")

st = st.sort_values(["trip_id", "sequence"])

def time_seconds(series):
    valid = series.str.fullmatch(r"\d{2,}:[0-5]\d:[0-5]\d")
    parts = series.where(valid).str.extract(
        r"^(\d+):(\d{2}):(\d{2})$"
    ).apply(pd.to_numeric, errors="coerce")

    return parts[0] * 3600 + parts[1] * 60 + parts[2]

for column in ["arrival_time", "departure_time"]:
    st[f"{column}_seconds"] = time_seconds(st[column])

    blank = st[column].str.strip().eq("")
    malformed = ~blank & st[f"{column}_seconds"].isna()

    note(
        f"stop_times: blank {column}",
        blank.sum(),
        "Preserved as missing; permitted cases require GTFS review."
    )
    note(
        f"stop_times: malformed {column}",
        malformed.sum(),
        "Excluded from time-based calculations."
    )

note(
    "stop_times: departure before arrival",
    (
        st["departure_time_seconds"]
        < st["arrival_time_seconds"]
    ).sum(),
    "Flagged for investigation; no automatic correction."
)

previous_departure = st.groupby("trip_id")[
    "departure_time_seconds"
].shift()

note(
    "stop_times: arrival before previous departure",
    (st["arrival_time_seconds"] < previous_departure).sum(),
    "Flagged for investigation; no automatic correction."
)

first = st.drop_duplicates("trip_id", keep="first")[
    ["trip_id", "departure_time", "departure_time_seconds"]
].rename(columns={"departure_time_seconds": "departure_seconds"})

last = st.drop_duplicates("trip_id", keep="last")[
    ["trip_id", "arrival_time_seconds"]
].rename(columns={"arrival_time_seconds": "last_arrival_seconds"})

patterns = (
    st.groupby("trip_id")["stop_id"]
    .agg(tuple)
    .reset_index(name="stop_pattern")
)

trip_features = (
    active_trips
    .merge(first, on="trip_id", how="left", validate="one_to_one")
    .merge(last, on="trip_id", how="left", validate="one_to_one")
    .merge(patterns, on="trip_id", how="left", validate="one_to_one")
)

pattern_lookup = {}
pattern_ids = []

for pattern in trip_features["stop_pattern"]:
    if not isinstance(pattern, tuple):
        pattern_ids.append(pd.NA)
    else:
        if pattern not in pattern_lookup:
            pattern_lookup[pattern] = f"pattern_{len(pattern_lookup) + 1}"
        pattern_ids.append(pattern_lookup[pattern])

trip_features["pattern_id"] = pattern_ids

trip_features["service_hour"] = (
    trip_features["departure_seconds"] // 3600
).astype("Int64")

trip_features["clock_hour"] = trip_features["service_hour"] % 24

trip_features["minute"] = (
    trip_features["departure_seconds"] % 3600 // 60
).astype("Int64")

def period(hour):
    if pd.isna(hour):
        return pd.NA
    if 5 <= hour < 12:
        return "Morning"
    if 12 <= hour < 17:
        return "Afternoon"
    if 17 <= hour < 21:
        return "Evening"
    return "Night"

trip_features["time_period"] = (
    trip_features["clock_hour"].apply(period)
)

trip_features["service_date"] = SERVICE_DATE.strftime("%Y-%m-%d")
trip_features["day_of_week"] = SERVICE_DATE.day_name()
trip_features["day_type"] = (
    "Weekend" if SERVICE_DATE.dayofweek >= 5 else "Weekday"
)

duration = (
    trip_features["last_arrival_seconds"]
    - trip_features["departure_seconds"]
) / 60

note(
    "trips: negative endpoint duration",
    duration.lt(0).sum(),
    "Negative duration excluded from duration summaries."
)

trip_features["scheduled_duration_minutes"] = duration.where(
    duration.ge(0)
)

note(
    "trips: missing or invalid first departure",
    trip_features["departure_seconds"].isna().sum(),
    "Trip retained in counts; excluded from departure summaries."
)

# --------------------------------------------------
# 5. ESTIMATE DISTANCES FROM ORDERED SHAPE POINTS
# --------------------------------------------------

shape_rows = []
shapes = tables.get("shapes")

if shapes is not None and "shape_id" in trip_features:
    for shape_id, group in shapes.groupby("shape_id"):
        group = group.copy()

        for column in [
            "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"
        ]:
            group[column] = pd.to_numeric(
                group[column], errors="coerce"
            )

        seq = group["shape_pt_sequence"]

        valid = (
            group["shape_pt_lat"].between(-90, 90)
            & group["shape_pt_lon"].between(-180, 180)
            & seq.notna()
            & seq.ge(0)
            & seq.mod(1).eq(0)
        )

        usable = (
            bool(str(shape_id).strip())
            and len(group) >= 2
            and valid.all()
            and not seq.duplicated().any()
        )

        distance_km = np.nan

        if usable:
            group = group.sort_values("shape_pt_sequence")
            lat = np.radians(group["shape_pt_lat"].to_numpy())
            lon = np.radians(group["shape_pt_lon"].to_numpy())

            dlat = np.diff(lat)
            dlon = np.diff(lon)

            a = (
                np.sin(dlat / 2) ** 2
                + np.cos(lat[:-1])
                * np.cos(lat[1:])
                * np.sin(dlon / 2) ** 2
            )

            distance_km = float(
                (
                    6371.0088 * 2
                    * np.arctan2(
                        np.sqrt(np.clip(a, 0, 1)),
                        np.sqrt(1 - np.clip(a, 0, 1))
                    )
                ).sum()
            )

        shape_rows.append({
            "shape_id": shape_id,
            "estimated_shape_length_km": distance_km,
            "usable_shape": usable
        })

shape_distances = pd.DataFrame(
    shape_rows,
    columns=["shape_id", "estimated_shape_length_km", "usable_shape"]
)

if not shape_distances.empty:
    trip_features = trip_features.merge(
        shape_distances[["shape_id", "estimated_shape_length_km"]],
        on="shape_id",
        how="left",
        validate="many_to_one"
    )

    note(
        "shapes: unusable shape groups",
        (~shape_distances["usable_shape"]).sum(),
        "Whole affected shape excluded; points were not silently dropped."
    )

# --------------------------------------------------
# 6. HEADWAYS FOR COMPARABLE TRIPS
# --------------------------------------------------

headway_keys = ["route_id", "pattern_id"]

if "direction_id" in trip_features:
    headway_keys.append("direction_id")

comparable = trip_features.dropna(
    subset=["departure_seconds", "pattern_id"]
).sort_values(headway_keys + ["departure_seconds"])

comparable = comparable.copy()
comparable["headway_minutes"] = (
    comparable.groupby(headway_keys)["departure_seconds"]
    .diff() / 60
)

headway_summary = (
    comparable.groupby(headway_keys)
    .agg(
        departures=("trip_id", "nunique"),
        interval_count=("headway_minutes", "count"),
        median_headway_minutes=("headway_minutes", "median"),
        min_headway_minutes=("headway_minutes", "min"),
        max_headway_minutes=("headway_minutes", "max")
    )
    .reset_index()
)

headway_summary["service_date"] = SERVICE_DATE.strftime("%Y-%m-%d")

# --------------------------------------------------
# 7. ROUTE PROFILES AND HOURLY COUNTS
# --------------------------------------------------

route_visits = st.merge(
    active_trips[["trip_id", "route_id"]],
    on="trip_id",
    validate="many_to_one"
)

trip_counts = (
    active_trips.groupby("route_id")["trip_id"]
    .nunique()
    .rename("scheduled_trip_count")
    .reset_index()
)

stop_counts = (
    route_visits.groupby("route_id")["stop_id"]
    .nunique()
    .rename("distinct_stop_count")
    .reset_index()
)

summary = (
    trip_features.groupby("route_id")
    .agg(
        trips_with_valid_departure=("departure_seconds", "count"),
        first_departure_seconds=("departure_seconds", "min"),
        last_departure_seconds=("departure_seconds", "max"),
        median_scheduled_duration_minutes=(
            "scheduled_duration_minutes", "median"
        )
    )
    .reset_index()
)

profiles = (
    routes
    .merge(trip_counts, on="route_id", how="left")
    .merge(stop_counts, on="route_id", how="left")
    .merge(summary, on="route_id", how="left")
)

profiles["scheduled_trip_count"] = (
    profiles["scheduled_trip_count"].fillna(0).astype(int)
)

profiles["distinct_stop_count"] = (
    profiles["distinct_stop_count"].astype("Int64")
)

profiles["departure_span_hours"] = (
    profiles["last_departure_seconds"]
    - profiles["first_departure_seconds"]
) / 3600

def gtfs_time(value):
    if pd.isna(value):
        return pd.NA
    value = int(value)
    return (
        f"{value // 3600:02d}:"
        f"{value % 3600 // 60:02d}:"
        f"{value % 60:02d}"
    )

for label in ["first", "last"]:
    profiles[f"{label}_departure_gtfs"] = (
        profiles[f"{label}_departure_seconds"].apply(gtfs_time)
    )

if "estimated_shape_length_km" in trip_features:
    lengths = (
        trip_features.groupby("route_id")
        .agg(
            trips_with_shape_distance=("estimated_shape_length_km", "count"),
            median_trip_shape_length_km=(
                "estimated_shape_length_km", "median"
            ),
            min_trip_shape_length_km=(
                "estimated_shape_length_km", "min"
            ),
            max_trip_shape_length_km=(
                "estimated_shape_length_km", "max"
            )
        )
        .reset_index()
    )
    profiles = profiles.merge(lengths, on="route_id", how="left")

profiles["service_date"] = SERVICE_DATE.strftime("%Y-%m-%d")
profiles["day_of_week"] = SERVICE_DATE.day_name()
profiles["day_type"] = (
    "Weekend" if SERVICE_DATE.dayofweek >= 5 else "Weekday"
)

hourly = (
    trip_features.dropna(subset=["service_hour"])
    .groupby(["route_id", "service_hour"])["trip_id"]
    .nunique()
    .reset_index(name="scheduled_departures")
)

hourly["service_date"] = SERVICE_DATE.strftime("%Y-%m-%d")

period_counts = (
    trip_features.dropna(subset=["time_period"])
    .groupby(["route_id", "time_period"])["trip_id"]
    .nunique()
    .reset_index(name="scheduled_departures")
)

period_counts["service_date"] = SERVICE_DATE.strftime("%Y-%m-%d")

# --------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------

assert profiles["scheduled_trip_count"].sum() == (
    active_trips["trip_id"].nunique()
)

assert hourly["scheduled_departures"].sum() == (
    trip_features["departure_seconds"].notna().sum()
)

assert period_counts["scheduled_departures"].sum() == (
    trip_features["departure_seconds"].notna().sum()
)

assert not profiles.duplicated(["route_id", "service_date"]).any()

# --------------------------------------------------
# 9. SAVE DATASETS
# --------------------------------------------------

trip_export = trip_features.drop(columns=["stop_pattern"]).copy()

pattern_export = pd.DataFrame([
    {
        "pattern_id": pattern_id,
        "ordered_stop_ids": json.dumps(list(pattern))
    }
    for pattern, pattern_id in pattern_lookup.items()
])

datasets = {
    "route_profiles": profiles,
    "trip_features": trip_export,
    "hourly_departures": hourly,
    "period_departures": period_counts,
    "headways_by_pattern": headway_summary,
    "stop_patterns": pattern_export,
    "shape_distances": shape_distances
}

for name, df in datasets.items():
    df.to_csv(OUTPUT / f"vapi_{name}.csv", index=False)

pd.DataFrame(inventory).to_csv(
    OUTPUT / "dataset_inventory.csv", index=False
)

pd.DataFrame(quality).to_csv(
    OUTPUT / "data_quality_report.csv", index=False
)

# --------------------------------------------------
# 10. FEATURE DICTIONARY AND DOCUMENTATION
# --------------------------------------------------

definitions = {
    "scheduled_trip_count": (
        "Distinct trips active on the selected service date",
        "trips and calendars", "Count distinct trip_id", "trips"
    ),
    "distinct_stop_count": (
        "Unique stops visited by active route trips",
        "trips and stop_times", "Count distinct stop_id per route", "stops"
    ),
    "departure_seconds": (
        "First-stop departure measured from service-day start",
        "stop_times", "hours*3600 + minutes*60 + seconds", "seconds"
    ),
    "service_hour": (
        "Departure hour preserving values above 23",
        "first-stop departure", "floor(departure_seconds / 3600)", "hours"
    ),
    "clock_hour": (
        "Clock hour from 0 to 23",
        "service_hour", "service_hour modulo 24", "hours"
    ),
    "minute": (
        "Minute component of first departure",
        "departure_seconds", "floor((seconds modulo 3600) / 60)", "minutes"
    ),
    "time_period": (
        "Morning, Afternoon, Evening or Night",
        "clock_hour", "Apply documented category boundaries", "category"
    ),
    "scheduled_departures": (
        "Distinct trip starts in the specified hour or period",
        "trip_features", "Group by route and hour/period; count distinct trips",
        "trips"
    ),
    "departure_span_hours": (
        "Span between earliest and latest trip starts",
        "first departures", "(latest - earliest) / 3600", "hours"
    ),
    "scheduled_duration_minutes": (
        "Scheduled first-departure to last-arrival duration",
        "stop_times", "(last arrival - first departure) / 60", "minutes"
    ),
    "estimated_shape_length_km": (
        "Estimated length of the complete linked shape",
        "shapes", "Sum Haversine distances between ordered points", "km"
    ),
    "median_trip_shape_length_km": (
        "Median linked shape length across active trips",
        "trip shape estimates", "Trip-weighted median within route", "km"
    ),
    "min_trip_shape_length_km": (
        "Shortest available linked trip shape",
        "trip shape estimates", "Minimum within route", "km"
    ),
    "max_trip_shape_length_km": (
        "Longest available linked trip shape",
        "trip shape estimates", "Maximum within route", "km"
    ),
    "headway_minutes": (
        "Gap between comparable consecutive first departures",
        "trip_features",
        "Sort within route, ordered stop pattern and available direction; diff/60",
        "minutes"
    )
}

dictionary = []

for dataset, df in datasets.items():
    for column in df.columns:
        if column in definitions:
            meaning, source, calculation, unit = definitions[column]
        elif column in routes.columns or column in trips.columns:
            meaning = f"Original GTFS field: {column}"
            source = "routes or trips"
            calculation = "Copied from source; IDs remain identifiers"
            unit = "source field"
        elif column in [
            "median_headway_minutes", "min_headway_minutes",
            "max_headway_minutes"
        ]:
            meaning = f"{column.replace('_', ' ')} for comparable departures"
            source = "first-stop departures"
            calculation = "Summary of consecutive departure gaps"
            unit = "minutes"
        else:
            meaning = column.replace("_", " ")
            source = "Selected date or derived summaries"
            calculation = {
                "service_date": "Selected analysis date",
                "day_of_week": "Weekday name of selected service date",
                "day_type": "Saturday/Sunday = Weekend; otherwise Weekday",
                "pattern_id": "Identifier for the exact ordered stop sequence",
                "ordered_stop_ids": "Ordered stop sequence serialized as JSON",
                "first_departure_seconds": "Minimum first departure within route",
                "last_departure_seconds": "Maximum first departure within route",
                "first_departure_gtfs": "Format earliest seconds as GTFS HH:MM:SS",
                "last_departure_gtfs": "Format latest seconds as GTFS HH:MM:SS",
                "trips_with_valid_departure": "Count usable first departures",
                "median_scheduled_duration_minutes": "Median usable endpoint duration",
                "trips_with_shape_distance": "Count trips with usable shape estimate",
                "departures": "Count distinct comparable trips",
                "interval_count": "Count available consecutive departure gaps",
                "usable_shape": "All shape checks passed and at least two points"
            }.get(column, "See program calculation")
            unit = "seconds" if column.endswith("_seconds") else (
                "minutes" if column.endswith("_minutes") else "not applicable"
            )

        dictionary.append({
            "dataset": dataset,
            "feature": column,
            "meaning": meaning,
            "source": source,
            "calculation": calculation,
            "unit": unit,
            "pandas_dtype": str(df[column].dtype)
        })

pd.DataFrame(dictionary).to_csv(
    OUTPUT / "feature_dictionary.csv", index=False
)

limitations = f"""
# Assumptions and limitations

- Feed: Vapi_GTFS, supplied in the uploaded archive.
- Analysis service date: {SERVICE_DATE.date()}.
- The original download source and licence must be verified separately.
- Only one service date is analysed; weekday/weekend comparisons require more dates.
- Morning: 05:00–11:59; Afternoon: 12:00–16:59;
  Evening: 17:00–20:59; Night: all remaining clock hours.
- GTFS hours above 24 are preserved as service-day hours.
- Hourly counts represent trip starts at the first listed stop.
- Missing first departures are not replaced by later-stop times.
- Hourly output contains observed departure hours only.
- Headways are grouped by route and exact ordered stop pattern,
  plus direction when available. Different stop patterns are kept separate.
- Single-departure groups have no measurable headway.
- Shape distances use ordered geographical points and Haversine distances.
  They are estimates of the complete linked shapes, not surveyed road distances.
- Shape jumps or incomplete geometry can distort estimates and need visual review.
- Route distance summaries use available trip shapes and may have partial coverage.
- Departure span measures trip-start spread, not continuous service duration.
- Endpoint durations do not validate every intermediate schedule value.
- Missing measurements are preserved. Zero trip count means no active trips.
- These checks are not a complete GTFS validator.
- No passenger counts, occupancy or actual traffic measurements are available.
- Scheduled service levels alone cannot prove overcrowding or underutilisation.
- No machine learning model is built in Module 2.
"""

(OUTPUT / "assumptions_limitations.md").write_text(
    limitations.strip(), encoding="utf-8"
)

readme = """
# RouteRanker Module 2 — Vapi

Run the program in Google Colab and upload the GTFS CSV ZIP.
Change SERVICE_DATE only to a date supported by the feed.

Outputs:
- vapi_route_profiles.csv: route features for the selected date.
- vapi_trip_features.csv: trip-level time and available distance features.
- vapi_hourly_departures.csv: departures by route and service-day hour.
- vapi_period_departures.csv: departures by time period.
- vapi_headways_by_pattern.csv: comparable departure-gap summaries.
- vapi_stop_patterns.csv: ordered stop-pattern definitions.
- vapi_shape_distances.csv: shape estimates and usability flags.
- dataset_inventory.csv: source structure and initial quality counts.
- data_quality_report.csv: findings and handling decisions.
- feature_dictionary.csv: output feature descriptions.
- assumptions_limitations.md: interpretation and limitations.

Keep the original archive and raw files separately.
Download the executed notebook from Colab as an additional deliverable.
"""

(OUTPUT / "README.md").write_text(
    readme.strip(), encoding="utf-8"
)

# --------------------------------------------------
# 11. DISPLAY RESULTS AND DOWNLOAD
# --------------------------------------------------

print("\nMODULE 2 CALCULATIONS COMPLETED")
print("Service date:", SERVICE_DATE.date())
print("Routes:", len(profiles))
print("Active trips:", active_trips["trip_id"].nunique())
print("Hourly total:", hourly["scheduled_departures"].sum())

display(profiles.head(10))

findings = pd.DataFrame(quality)
findings = findings[findings["affected_records"] > 0]

print("\nQUALITY FINDINGS")
if findings.empty:
    print("No findings in the recorded checks.")
else:
    display(findings)

zip_path = shutil.make_archive(
    "/content/RouteRanker_Module2_Vapi",
    "zip",
    root_dir=str(OUTPUT)
)

files.download(zip_path)

print("\nDownload the executed notebook separately:")
print("File → Download → Download .ipynb")

Upload All_GTFS_CSV_Files.zip


Saving All_GTFS_CSV_Files-1.zip to All_GTFS_CSV_Files-1 (1).zip

MODULE 2 CALCULATIONS COMPLETED
Service date: 2026-10-31
Routes: 22
Active trips: 461
Hourly total: 461


,route_id,agency_id,route_short_name,route_long_name,route_type,scheduled_trip_count,distinct_stop_count,trips_with_valid_departure,first_departure_seconds,last_departure_seconds,...,departure_span_hours,first_departure_gtfs,last_departure_gtfs,trips_with_shape_distance,median_trip_shape_length_km,min_trip_shape_length_km,max_trip_shape_length_km,service_date,day_of_week,day_type
0,1868,VCBS,11,"Vapi Town Post office - Meril, Welspun Company...",3,31,13,31,44820,76620,...,8.833333,12:27:00,21:17:00,31,13.040084,13.040084,13.040084,2026-10-31,Saturday,Weekend
1,1867,VCBS,10,"Vapi Town Post office - Dabhel Check Post, Kun...",3,16,4,16,46800,76500,...,8.25,13:00:00,21:15:00,16,2.884072,2.884072,2.884072,2026-10-31,Saturday,Weekend
2,1829,VCBS,1,Vapi Railway Station (East) - G.I.D.C. Third P...,3,19,5,19,45540,70380,...,6.9,12:39:00,19:33:00,19,6.354314,6.354314,6.354314,2026-10-31,Saturday,Weekend
3,1890,VCBS,6,"Gunjan Intersection , Morarji Circle , Godal N...",3,9,7,9,46500,73380,...,7.466667,12:55:00,20:23:00,9,8.280020,8.280020,8.280020,2026-10-31,Saturday,Weekend
4,1833,VCBS,3,Vapi Railway Station (East) - Chhiri,3,21,5,21,43980,71580,...,7.666667,12:13:00,19:53:00,21,10.607384,10.607384,10.607384,2026-10-31,Saturday,Weekend
5,1830,VCBS,1,G.I.D.C. Third Phase - Vapi Railway Station (E...,3,51,2,51,44160,72960,...,8.0,12:16:00,20:16:00,51,1.577181,1.577181,1.577181,2026-10-31,Saturday,Weekend
6,1881,VCBS,5,"Vapi Railway Station(West) - Roffel College, N...",3,9,5,9,45300,72180,...,7.466667,12:35:00,20:03:00,9,2.448396,2.448396,2.448396,2026-10-31,Saturday,Weekend
7,1882,VCBS,5,"Kachigam Gate , Chandor, Namdha ,Roffel Colleg...",3,9,2,9,46980,73860,...,7.466667,13:03:00,20:31:00,9,0.196766,0.196766,0.196766,2026-10-31,Saturday,Weekend
8,1889,VCBS,6,"Vapi Railway Station (East) ,Godal Nagar, Mora...",3,9,7,9,44820,71700,...,7.466667,12:27:00,19:55:00,9,7.824221,7.824221,7.824221,2026-10-31,Saturday,Weekend
9,1831,VCBS,2,Vapi Railway Station (East) - Dadra check post,3,24,8,24,44820,70200,...,7.05,12:27:00,19:30:00,24,8.702078,8.702078,8.702078,2026-10-31,Saturday,Weekend



QUALITY FINDINGS
No findings in the recorded checks.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download the executed notebook separately:
File → Download → Download .ipynb
